<a href="https://colab.research.google.com/github/Text-Machine/mask-predict/blob/main/1-compute-gradients.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/> </a>

# Compute Gradients

This notebooks is prepared for computing gradients given a preformatted csv file.

Gradients compute the relation between context words and a given target word, which in our case is the token we masked, and which we want to investigate. This masked token is also referred to as the target token.

This notebooks is written for Colab but should also work on other platforms with some minor adjustments.

To measure the influence between context and target words, we need a sentences with a [MASKED] token, and a set of target tokens, for which we want to compute the gradients in the masked token position.

There are two main options strategies, and these options relate to how we model the set of target tokens

## Predictions

Here we model the relationship between contexts words and tokens predicted by a BERT model using in the fill-mask pipeline. 

### Predictions-filtered

This is a subtask, where we narrow down the set of target tokens by thematically filtering the predicted words, for example, only keep examples where BERT predicted human-like words. This filtering can happen iteratively.

## Contrastive

We provide a contrastive pair of keywords, e.g. 'machine' and 'slave' and for each senteces we compute gradients for the same pair.



In [ ]:
#!git clone https://github.com/Text-Machine/mask-predict.git

In [ ]:
#%cd mask-predict

In [ ]:
#!pip install  -q -e .

# Prepare variables and load data

In [ ]:
from explain import *
from collections import Counter
import pandas as pd
from pathlib import Path
import json

In [ ]:
collection,genre_suffix = 'blb',''
if collection == 'blb':
  genre_suffix = '_with_genre'

maskedToken = 'machine'

modelName = "Livingwithmachines/bert_1760_1900"
dataPath = 'input_data'
processedFolder = Path('gradient_data')
processedFolder.mkdir(exist_ok=True, parents=True)
predCol = "pred_bert_1760_1900"
iloc_range = (0, -1)

In [ ]:
#%cd /content

In this notebook we analyse the newspaper sentences where machine in mentioned by BLERT predicts words related to slavery.

In [ ]:
#!unzip -o "{collection}_{maskedToken}_clusters{genre_suffix}_deduplicated.tsv.zip"

We rank sentences by their similarity to the 'slave' cluster based on the BLERT predictions. 

In [ ]:

data_df = pd.read_csv(f"./{dataPath}/{collection}_{maskedToken}_clusters{genre_suffix}_deduplicated.tsv", sep="\t", index_col=0)
data_df.head()

In [ ]:
data_df.tail(3)

We get all the sentences, and the top 5 predictions. This means we only look at context words where our target words of interest (like 'slave' appear in the top five words).

# Thematic filtering of the predicted target

To minimise the computation we focus on a specific sample of predictions, i.e. occurances where the MLM predicted human type words instead of machine words. 

We start with a seed list of predictions, look which other words are predicted together with these seedwords and use this expanded list as our main target of investigation.

In [ ]:
texts, predicted_targets = build_texts_targets(
    data_df, start=iloc_range[0], end=iloc_range[1],
    pred_col=predCol, top_n=5
)

In [ ]:
def filter_texts_targets(texts, predicted_targets, kw_filters, data_df, only_keywords=False):
    filtered_texts = []
    filtered_predicted_targets = []
    filter_identifiers = []
    for i,(text, preds) in enumerate(zip(texts, predicted_targets)):
        if set(preds).intersection(set(kw_filters)):
            filtered_texts.append(text)
            filter_identifiers.append(i)
            if only_keywords:
                preds = [p for p in preds if p in kw_filters]
                if not preds:
                    continue
            filtered_predicted_targets.append([data_df.targetExpression.iloc[i]]+preds)
    return filter_identifiers, filtered_texts, filtered_predicted_targets

In the example we first fetch human words by looking which other prediction co-occur with a set of the seed words such 'child', 'man' etc. The idea is the obtain an expanded list of words to investigate, and ensure our sample isn't completely biased by the 
We use `filter_texts_targets`, in the first round we set the  `only_keywords` argument to `False`, in the second round to `True` because we want to work only with the expanded wordlist.

In [ ]:
#kw_filters = ['child', 'children', 'woman', 'man', 'women', 'men', 'slave','slaves']
kw_filters = open(f'{dataPath}/250_freq_pred_KB_edit.txt').read().splitlines()
kw_filters[:4]

In [ ]:

print('Before filtering:', len(texts), len(predicted_targets))
filter_identifiers, texts, predicted_targets = filter_texts_targets(texts, 
                                                                    predicted_targets, 
                                                                    kw_filters, data_df,
                                                                    only_keywords=True)
print('After filtering:', len(texts), len(predicted_targets))

In [ ]:
data_df.iloc[filter_identifiers].to_csv(f"{processedFolder}/{collection}_{maskedToken}{genre_suffix}_kw_filtered.tsv", sep="\t")


In [ ]:
# from google.colab import files
# files.download(f"{processedFolder}/{collection}_{maskedToken}_filtered{genre_suffix}_kw_filtered.tsv") 

In [ ]:
Counter([t for l in predicted_targets for t in l]).most_common(10)

In [ ]:
open(f'{dataPath}/250_freq_pred.txt','w').write('\n'.join([s for s,i in Counter([t for l in predicted_targets for t in l]).most_common(250)]))

In [ ]:
idx = 100
texts[idx],predicted_targets[idx], filter_identifiers[idx]

We run the explainer using BLERT as the model we inspect, our targets are the top 5 predictions, the texts are the 1000 sentences selected in the previous cell.

# Compute Integrated Gradients.

In [ ]:
explainer = MaskedLMExplainer(model_name=modelName, device=pick_device())

In [ ]:
results = explainer.explain(texts, predicted_targets, word_agg="max")

In [ ]:
with open(f'{processedFolder}/results_{collection}_{maskedToken}_pred_kw_filtered.json', 'w') as f:
    json.dump(results, f)

In [ ]:
# with open(f'{processedFolder}/results_{collection}_{maskedToken}_pred_kw_filtered.json', 'r') as f:
#     results = json.load(f)

In [ ]:
# from google.colab import files
# files.download(f'results_{collection}_{maskedToken}_pred_kw_filtered.json') 

Only retain results for context word contribution when slave or machine are predicted by BLERT.

In [ ]:
results_df = result_as_dataframe(results, kw_filters + list(data_df.targetExpression.unique()))
#results_df['identifier'] = results_df.id.apply(lambda x: filter_identifiers[int(x)])
results_df.to_csv(f'{processedFolder}/results_{collection}_{maskedToken}_pred_kw_filtered.csv')

In [ ]:
#files.download(f'results_{collection}_{maskedToken}_pred_kw_filtered.csv') 

# Contrastive Analysis

Contrastive analysis, we use slave and machine as target words for all sentences, ignoring the actual predictions.

In [ ]:
if maskedToken == 'slave':
      contrastive_dict = {'slave':'machine', 'slaves':'machines'}
elif maskedToken == 'machine':
    contrastive_dict = {'machine':'slave', 'machines':'slaves'}

data_df['targetExpressionProccessed'] = data_df['targetExpression'].apply(lambda x: x.lower().strip())
data_df['targetContrExpressionProccessed'] = data_df['targetExpressionProccessed'].apply(lambda x: contrastive_dict.get(x, x))
contrastive_targets = data_df.iloc[iloc_range[0]:iloc_range[1]][['targetContrExpressionProccessed', 'targetExpressionProccessed']].values.tolist()

len(texts) == len(contrastive_targets)

In [ ]:
contrastive_targets[:3]

In [ ]:
results2 = explainer.explain(texts, contrastive_targets, word_agg="max")

In [ ]:
with open(f'results_{collection}_{maskedToken}_constrastive.json', 'w') as f:
    json.dump(results2, f)

In [ ]:
with open(f'{processedFolder}/results_{collection}_{maskedToken}_constrastive.json', 'r') as f:
    results2 = json.load(f)

In [ ]:
unique = list(set(x for sublist in contrastive_targets for x in sublist))
unique

In [ ]:
results_df = result_as_dataframe(results2,unique)
results_df.shape

In [ ]:
results_df.to_csv(f'{processedFolder}/results_{collection}_{maskedToken}_constrastive.csv')

# Fin.